### Descrição da Etapa de Pré-processamento (Metodologia)

A etapa de pré-processamento do *corpus* textual consistiu em um pipeline de normalização e limpeza de dados visando a padronização das estruturas sintáticas e a integridade semântica dos documentos. Inicialmente, procedeu-se à remoção de ruídos textuais, especificamente *tags* HTML, por meio de expressões regulares. Em seguida, aplicou-se a padronização de espaçamentos, compreendendo a eliminação de espaços múltiplos e a supressão de espaços indevidos antes de vírgulas. De modo a preservar a morfologia de terminologias e abreviações próprias do domínio jurídico — como "art." (artigo), "c.c." (combinado com) e "n." (número) —, implementou-se uma heurística de consolidação que protege esses termos contra regras genéricas de tokenização e espaçamento de pontuação. Para tal, utilizou-se uma estratégia de substituição temporária por marcadores (*placeholders*), garantindo que a regra de inserção de espaço após pontos finais fosse aplicada exclusivamente para delimitação de sentenças, não afetando os acrônimos jurídicos mapeados nem a pontuação interna de valores numéricos.

In [87]:
from nltk.tokenize import word_tokenize
import re
import nltk

try:
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except Exception as e:
    print(f"Aviso: falha ao baixar recursos NLTK: {e}")

# 1. Definindo a lista de stopwords base com as customizações necessárias
portuguese_stopwords_list = set(nltk.corpus.stopwords.words('portuguese'))
portuguese_stopwords_list.update(['e', 'é', 'o', 'a', 'os', 'as', 'nesta', 'neste', 'desse', 'dessa'])

# ---------------------------------------------------------------------------
# Mapa centralizado de abreviações jurídicas protegidas
# Evita duplicação de placeholders entre remove_whitespace e remove_pontuacao
# ---------------------------------------------------------------------------
_SIMPLE_ABBREVS = [
    'art', 'n', 'fls', 'fl', 'inc', 'min', 'des', 'rel',
    'cf', 'doc', 'proc', 'dr', 'dra',
]
_DIGIT_DOT_PH = "_#_DIGIT_DOT_#_"
_CC_DOT_PH = "_#_CC_DOT_#_"   # c.c.
_CC_PH = "_#_CC_#_"           # c.c (sem ponto final)
_NRO_PH = "_#_NRO_#_"         # nº


def _abbr_ph(abbr):
    """Gera placeholder para uma abreviação simples."""
    return f"_#_{abbr.upper()}_DOT_#_"


def remove_noise(text):
    if isinstance(text, str):
        clean = re.compile('<.*?>')
        return re.sub(clean, ' ', text)
    return text


def _consolidate_known_acronyms(text):
    """Junta abreviações cujo ponto ficou separado por espaço (ex.: 'art .' -> 'art.')."""
    text = re.sub(r'(?i)(\bart)\s+\.', r'\1.', text)
    text = re.sub(r'(?i)(\bn)\s+\.', r'\1.', text)
    text = re.sub(r'(?i)(c)\s+\.\s*(c)\s+\.', r'\1.\2.', text)
    text = re.sub(r'(?i)(c)\s+\.\s*(c)', r'\1.\2', text)
    text = re.sub(r'(?i)(c\.)\s+(c)', r'\1\2', text)
    for abbr in _SIMPLE_ABBREVS:
        if abbr not in ('art', 'n'):
            text = re.sub(r'(?i)(\b' + re.escape(abbr) + r')\s+\.', r'\1.', text)
    return text


def _protect_dots(text):
    """Substitui pontos de abreviações jurídicas e dígitos por placeholders."""
    # Pontos entre dígitos (ex.: 11.343)
    text = re.sub(r'(\d)\.(\d)', r'\1' + _DIGIT_DOT_PH + r'\2', text)
    # nº (protege antes que º seja removido como pontuação)
    text = re.sub(r'(?i)\bnº', _NRO_PH, text)
    # Compostos: c.c. e c.c (substituição completa para proteger AMBOS os pontos)
    text = re.sub(r'(?i)\bc\.c\.', _CC_DOT_PH, text)
    text = re.sub(r'(?i)\bc\.c(?!\w)', _CC_PH, text)
    # Abreviações simples
    for abbr in _SIMPLE_ABBREVS:
        ph = _abbr_ph(abbr)
        text = re.sub(r'(?i)(\b' + re.escape(abbr) + r')\.', r'\1' + ph, text)
    return text


def _restore_dots(text):
    """Restaura todos os placeholders para seus valores originais."""
    text = text.replace(_DIGIT_DOT_PH, '.')
    text = text.replace(_NRO_PH, 'nº')
    text = text.replace(_CC_DOT_PH, 'c.c.')
    text = text.replace(_CC_PH, 'c.c')
    for abbr in _SIMPLE_ABBREVS:
        text = text.replace(_abbr_ph(abbr), '.')
    return text


def remove_whitespace(text):
    if not isinstance(text, str):
        return text

    # 1. Replace multiple whitespace characters with a single space
    text = re.sub(r'\s+', ' ', text).strip()

    # 2. Consolidate specific acronym patterns
    text = _consolidate_known_acronyms(text)

    # 3. Remove space before commas and colons
    text = re.sub(r'\s+,', ',', text)
    text = re.sub(r'\s+:', ':', text)

    # 3.5 Remove spaces inside parentheses, brackets, and braces
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    text = re.sub(r'\[\s+', '[', text)
    text = re.sub(r'\s+\]', ']', text)
    text = re.sub(r'\{\s+', '{', text)
    text = re.sub(r'\s+\}', '}', text)

    # 3.6 Padronizar espaçamento do símbolo de parágrafo (§)
    text = re.sub(r'§\s*', '§ ', text)

    # 4. Proteger pontos de abreviações e dígitos
    text = _protect_dots(text)

    # Aplicar regra genérica: espaço após pontos restantes
    text = re.sub(r'\.(?=\S)', '. ', text)

    # Restaurar pontos protegidos
    text = _restore_dots(text)

    return text


def remove_space_before_last_period(text):
    """Remove espaço antes de qualquer ponto."""
    return re.sub(r' \.', '.', text)


def remove_pontuacao(texto):
    if not isinstance(texto, str):
        return texto

    # Proteger abreviações jurídicas e pontos numéricos
    texto = _protect_dots(texto)

    # Remover pontuação ASCII e Unicode comum em textos jurídicos
    # Inclui: . , ! ? ; : ( ) [ ] { } " ' ° ª º / \ – — « » "" '' -
    texto = re.sub(
        r'[.,!?;:()\[\]{}"\'°ªº/\\–—«»\u201c\u201d\u2018\u2019\-]',
        ' ',
        texto
    )

    # Restaurar abreviações protegidas
    texto = _restore_dots(texto)

    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


def pre_processamento(texto, mantem_pontuacao='S', stop_words=portuguese_stopwords_list):
    if not isinstance(texto, str):
        return ""

    # 1. Limpeza de HTML/Ruído
    texto = remove_noise(texto)

    # 2. Arrumar espaços ANTES de tokenizar
    texto = remove_whitespace(texto)
    texto = remove_space_before_last_period(texto)

    # 3. Tokenização
    tokens = word_tokenize(texto, language='portuguese')

    # 4. Filtro de Stopwords customizado
    filtered_tokens = [w for w in tokens if w.lower() not in stop_words or not w.isalpha()]

    # 5. Reconstrução do texto
    texto = " ".join(filtered_tokens)

    # 6. Nova limpeza de segurança para espaços e pontuação final
    texto = remove_whitespace(texto)
    texto = remove_space_before_last_period(texto)

    if mantem_pontuacao.strip().upper() != 'S':
        texto = remove_pontuacao(texto)
        texto = texto.lower()

    return texto

In [88]:
# Vamos verificar se 'e' está na lista padrão do NLTK
# print("'e' na lista do NLTK:", 'e' in portuguese_stopwords_list)

# # Podemos criar uma lista customizada adicionando mais palavras que você percebeu que não foram removidas
# custom_stopwords = set(portuguese_stopwords_list)
# custom_stopwords.update(['e', 'é', 'em', 'Em', 'o', 'a', 'os', 'as'])

# # E você pode atualizar a função pre_processamento para usar custom_stopwords no lugar de portuguese_stopwords_list
# print("Total de stopwords atualizadas:", len(custom_stopwords))

In [ ]:
# Testes de regressão do espaçamento antes do ponto
# casos = [
#     'O ponto final deveria estar aqui sem espaço.',
#     'Teste ponto. Ponto teste.',
# ]

# for texto_teste in casos:
#     resultado = pre_processamento(texto_teste, mantem_pontuacao='S', stop_words=set())
#     print(f'Input:  |{texto_teste}|')
#     print(f'Output: |{resultado}|')
#     print(f"Sem ' .'? {' .' not in resultado}")
#     print('-' * 60)

Input:  |O ponto final deveria estar aqui sem espaço.|
Output: |O ponto final deveria estar aqui sem espaço.|
Sem ' .'? True
------------------------------------------------------------
Input:  |Teste ponto. Ponto teste.|
Output: |Teste ponto. Ponto teste.|
Sem ' .'? True
------------------------------------------------------------


###Exemplo de uso das funções

In [90]:
#Exemplo de uso da função
# texto = """  <br><hr> Trata-se , de   habeas   corpus substitutivo   <b> de </b>
# recurso          próprio , com pedido de liminar.Porém, tudo de bom.De bom.1.1253,00</br> """
# texto = remove_noise(texto)

# print(texto)

In [91]:
# text = "583 dias-multa , incurso art. 33 , caput , c. c 40 , VI , Lei n . 11.343/2006.Em sede recursal , Tribunal origem negou provimento apelo defensivo"
# text = _consolidate_known_acronyms(text)
# print(text)

###Stopword removal


In [92]:
# text = "<br>583 dias-multa , incurso art . 33 , <b>caput , </b>c. c 40 , VI , Lei n . 11.343/2006.Em sede recursal , Tribunal origem negou provimento apelo defensivo."

# text = remove_noise(text)
# print(f"After remove_noise: {text}")

# text = remove_whitespace(text)
# print(f"After remove_whitespace: {text}")

# text = _consolidate_known_acronyms(text)
# print(f"After remove_whitespace: {text}")


In [95]:
#Exemplo de uso da função
texto = """<br>O ministro presidente e relator (SEBASTIÃO REIS): Trata-se de habeas: corpus <b>substitutivo de recurso próprio</b>, com pedido de liminar, impetrado em favor de LUIS FELIPE SILVA DE ASSIS, em que se aponta com autoridade coatora o Tribunal de Justiça do Estado do Rio de Janeiro.Consta dos autos que o paciente foi condenado à pena de 5 anos e 10 meses de reclusão, em regime semiaberto, mais pagamento de 583 dias-multa, como incurso no art. 33,caput, c.c o 40, VI, da Lei n. 11.343/2006.Em sede recursal, o Tribunal de origem negou provimento ao apelo defensivo."""
texto = pre_processamento(texto, 'S')

print(texto)

ministro presidente relator (SEBASTIÃO REIS): Trata-se habeas: corpus substitutivo recurso próprio, pedido liminar, impetrado favor LUIS FELIPE SILVA ASSIS, aponta autoridade coatora Tribunal Justiça Estado Rio Janeiro. Consta autos paciente condenado pena 5 anos 10 meses reclusão, regime semiaberto, pagamento 583 dias-multa, incurso art. 33, caput, c.c 40, VI, Lei n. 11.343/2006. sede recursal, Tribunal origem negou provimento apelo defensivo.


In [ ]:
print ("version 1.04.0")

version 1.03.1
